<a href="https://colab.research.google.com/github/rahulsvt-1907/Try-on/blob/Google-colab/Virtual_Try_on.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gradio as gr
from diffusers import StableDiffusionInpaintPipeline, ControlNetModel, StableDiffusionControlNetInpaintPipeline
from PIL import Image
import torch
import numpy as np
from transformers import AutoProcessor, AutoModel
import cv2
from huggingface_hub import login # Import login

In [ ]:
# Authenticate with Hugging Face Hub (optional, but recommended for higher rate limits)
# If you have an HF token, store it as a Colab secret named 'HF_TOKEN'
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print("✅ Logged into Hugging Face Hub!")
    else:
        print("⚠️ HF_TOKEN not found in Colab secrets. Proceeding unauthenticated.")
except Exception as e:
    print(f"⚠️ Could not log into Hugging Face Hub: {e}. Proceeding unauthenticated.")

# 🔥 Hugging Face Models (Best Combination)
CONTROL_NET_MODEL = "lllyasviel/control_v11p_sd15_inpaint"
BASE_MODEL = "runwayml/stable-diffusion-inpainting"
VITON_MODEL = "yisol/IDM-VTON"

In [ ]:
class HFVirtualTryOn:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.setup_models()

    def setup_models(self):
        """Initialize Hugging Face models"""
        try:
            # ControlNet for pose/clothing control
            self.controlnet = ControlNetModel.from_pretrained(
                CONTROL_NET_MODEL,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
            )

            # Stable Diffusion Inpainting
            self.pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
                BASE_MODEL,
                controlnet=self.controlnet,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
            )

            self.pipe = self.pipe.to(self.device)

            # Enable memory efficient attention
            if hasattr(self.pipe, "enable_attention_slicing"):
                self.pipe.enable_attention_slicing()

            print("✅ Models loaded successfully!")

        except Exception as e:
            print(f"⚠️ Model loading error: {e}")
            # Fallback to simple pipeline
            self.pipe = StableDiffusionInpaintPipeline.from_pretrained(
                BASE_MODEL,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
            )
            self.pipe = self.pipe.to(self.device)

    def create_mask(self, person_image, clothing_image):
        """Create mask for inpainting"""
        # Simple mask creation (replace with segmentation model in production)
        mask = Image.new('L', person_image.size, 0)

        # Create a basic mask for the clothing area
        person_np = np.array(person_image)
        clothing_np = np.array(clothing_image.resize(person_image.size))

        # Convert to grayscale and create difference mask
        person_gray = cv2.cvtColor(person_np, cv2.COLOR_RGB2GRAY)
        clothing_gray = cv2.cvtColor(clothing_np, cv2.COLOR_RGB2GRAY)

        # Simple threshold-based mask
        diff = cv2.absdiff(person_gray, clothing_gray)
        _, mask_np = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)

        mask = Image.fromarray(mask_np)
        return mask

    def generate_tryon(self, person_image, clothing_image, prompt="", strength=0.75):
        """Generate virtual try-on using Stable Diffusion"""
        try:
            # Resize images
            person_image = person_image.resize((512, 512))
            clothing_image = clothing_image.resize((512, 512))

            # Create mask
            mask = self.create_mask(person_image, clothing_image)

            # Prepare prompt
            if not prompt:
                prompt = "person wearing the clothing, natural lighting, high quality, detailed"

            # Generate using Stable Diffusion
            result = self.pipe(
                prompt=prompt,
                image=person_image,
                mask_image=mask,
                num_inference_steps=30,
                guidance_scale=7.5,
                strength=strength
            ).images[0]

            return result

        except Exception as e:
            print(f"Generation error: {e}")
            # Fallback to simple blending
            return self.simple_blend(person_image, clothing_image)

    def simple_blend(self, person_image, clothing_image):
        """Fallback blending method"""
        clothing_resized = clothing_image.resize(person_image.size)
        if clothing_resized.mode != 'RGBA':
            clothing_resized = clothing_resized.convert('RGBA')

        person_rgba = person_image.convert('RGBA')
        result = Image.blend(person_rgba, clothing_resized, alpha=0.5)
        return result.convert('RGB')


In [2]:
import gradio as gr
from diffusers import StableDiffusionInpaintPipeline, ControlNetModel, StableDiffusionControlNetInpaintPipeline
from PIL import Image
import torch
import numpy as np
from transformers import AutoProcessor, AutoModel
import cv2
from huggingface_hub import login # Import login



# Initialize the system
try_on_system = HFVirtualTryOn()

def virtual_try_on_gradio(person_image, clothing_image, prompt, strength):
    """Gradio interface function"""
    if person_image is None or clothing_image is None:
        return None

    result = try_on_system.generate_tryon(
        person_image,
        clothing_image,
        prompt,
        strength
    )

    return result

# 🚀 Gradio Interface
def create_gradio_interface():
    with gr.Blocks(title="🎯 AI Virtual Try-On") as demo:
        gr.Markdown("""
        # 🎯 AI Virtual Try-On System
        ### Powered by Stable Diffusion + ControlNet (Hugging Face)

        Upload a person image and clothing image to see the magic happen!
        """)

        with gr.Row():
            with gr.Column():
                person_input = gr.Image(
                    label="👤 Person Image",
                    type="pil",
                    height=300
                )
                clothing_input = gr.Image(
                    label="👕 Clothing Image",
                    type="pil",
                    height=300
                )

            with gr.Column():
                output_image = gr.Image(
                    label="✨ Result",
                    type="pil",
                    height=400
                )

                with gr.Accordion("⚙️ Advanced Settings", open=False):
                    prompt = gr.Textbox(
                        label="Prompt",
                        placeholder="Describe the desired result...",
                        value="person wearing the clothing, natural lighting, high quality"
                    )
                    strength = gr.Slider(
                        minimum=0.1,
                        maximum=1.0,
                        value=0.75,
                        step=0.05,
                        label="Strength"
                    )

        try_btn = gr.Button("✨ Generate Try-On", variant="primary")

        try_btn.click(
            fn=virtual_try_on_gradio,
            inputs=[person_input, clothing_input, prompt, strength],
            outputs=output_image
        )

    return demo

# 🚀 Launch the app
if __name__ == "__main__":
    demo = create_gradio_interface()
    demo.launch(
        server_name="0.0.0.0",
        server_port=None,
        share=True,  # Creates public URL
        show_error=True
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--run

✅ Models loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://05c82381c09a655229.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 📝 How to use:
1. Upload a clear person image
2. Upload a clothing image
3. Adjust settings if needed
4. Click "Generate Try-On"

### 🎯 Tips for best results:
- Use high-quality images
- Ensure good lighting
- Front-facing poses work best